## Supplemental analysis for bionomial modeling (life history)

In [60]:
fit_model_binomial <- function(data, Kmat,responseVar,predictor) {
#   library(asreml)
    gc()
    asreml.options(verbose = FALSE)
    Coeff = c()
    WaldPval = c()
    for (i in predictor){
        fullFM = as.formula(paste0(responseVar, "~", i ,"+ compdup + pacbio"))
        model_full <- asreml(fixed = fullFM,
                             random = ~ vm(assemblyID, Kmat) , ai.sing = F, data = data,family =  asr_binomial(link = "logit", dispersion = 1, total = NULL))

        modelWald <- wald.asreml(model_full)
        if(model_full$converge) {
            Coeff <- c(Coeff,model_full$coefficients$fixed[2])
            WaldPval <- c(WaldPval,modelWald[2,4])
        }else {
            Coeff <- c(Coeff,NA)
            WaldPval <- c(WaldPval,NA)
        }
    }
    
    out = c(Coeff,WaldPval)
    names(out) = c(paste("Coeff",predictor,sep = "_"),
                   paste("P",predictor,sep = "_"))
    return(out)
}

dat1_LH = subset(dat1,lifeHistory%in% c("annual", "perennial"))
dat1_LH$lifeHistoryNumeric = as.numeric(as.factor(dat1_LH$lifeHistory))-1
dat1_RZ = subset(dat1,lifeHistory%in% c("perennial") & !is.na(rhizome_PAV))

dat2_LH = subset(dat2,lifeHistory%in% c("annual", "perennial"))
dat2_LH$lifeHistoryNumeric = as.numeric(as.factor(dat2_LH$lifeHistory))-1
dat2_RZ = subset(dat2,lifeHistory%in% c("perennial")& !is.na(rhizome_PAV))

dat3_LH = subset(dat3,lifeHistory%in% c("annual", "perennial"))
dat3_LH$lifeHistoryNumeric = as.numeric(as.factor(dat3_LH$lifeHistory))-1
dat3_RZ = subset(dat3,lifeHistory%in% c("perennial")& !is.na(rhizome_PAV))

dat4_LH = subset(dat4,lifeHistory%in% c("annual", "perennial"))
dat4_LH$lifeHistoryNumeric = as.numeric(as.factor(dat4_LH$lifeHistory))-1
dat4_RZ = subset(dat4,lifeHistory%in% c("perennial")& !is.na(rhizome_PAV))

dat5_LH = subset(dat5,lifeHistory%in% c("annual", "perennial"))
dat5_LH$lifeHistoryNumeric = as.numeric(as.factor(dat5_LH$lifeHistory))-1
dat5_RZ = subset(dat5,lifeHistory%in% c("perennial")& !is.na(rhizome_PAV))

asreml.options(maxit = 60,verbose = F)
LHenvPC_MLM=fit_model_binomial(dat1_LH,phyloKMat,"lifeHistoryNumeric",colnames(dat1_LH)[31:325])
RZenvPC_MLM=fit_model_binomial(dat1_RZ,phyloKMat,"rhizome_PAV",colnames(dat1_LH)[31:325])

LHenvPC_MLMResTab = cbind(-LHenvPC_MLM[1:295]*correctionFactor,LHenvPC_MLM[-c(1:295)])
RZenvPC_MLMResTab = cbind(RZenvPC_MLM[1:295]*correctionFactor2,RZenvPC_MLM[-c(1:295)])

LHRZCol = rep("grey",nrow(LHenvPC_MLMResTab))
LHRZCol[LHenvPC_MLMResTab[,2]<0.05 & RZenvPC_MLMResTab[,2]>0.05 & LHenvPC_MLMResTab[,1]>0] = "orange"
LHRZCol[LHenvPC_MLMResTab[,2]<0.05 & RZenvPC_MLMResTab[,2]>0.05 & LHenvPC_MLMResTab[,1]<0] = "forestgreen"
LHRZCol[LHenvPC_MLMResTab[,2]>0.05 & RZenvPC_MLMResTab[,2]<0.05 & RZenvPC_MLMResTab[,1]>0] = "darkred"
LHRZCol[LHenvPC_MLMResTab[,2]>0.05 & RZenvPC_MLMResTab[,2]<0.05 & RZenvPC_MLMResTab[,1]<0] = "pink"
LHRZCol[LHenvPC_MLMResTab[,2]<0.05 & RZenvPC_MLMResTab[,2]<0.05 & RZenvPC_MLMResTab[,1]>0 & LHenvPC_MLMResTab[,1]*RZenvPC_MLMResTab[,1]<0] = "blue"
LHRZCol[LHenvPC_MLMResTab[,2]<0.05 & RZenvPC_MLMResTab[,2]<0.05 & RZenvPC_MLMResTab[,1]<0 & LHenvPC_MLMResTab[,1]*RZenvPC_MLMResTab[,1]<0] = "purple"


LHRZenv_MLMResTab = cbind(LHenvPC_MLMResTab,RZenvPC_MLMResTab,LHRZCol)
rownames(LHRZenv_MLMResTab) = gsub("Coeff_","",rownames(LHRZenv_MLMResTab))
colnames(LHRZenv_MLMResTab) = c("LifeHistory_Coeff","LifeHistory_pVal","Rhizome_Coeff","Rhizome_pVal","group")

write.table(LHRZenv_MLMResTab,"/workdir/sh2246/p_phyloGWAS/output/LHRZenv_MLMResTab.txt",quote = F,sep = "\t")

plot(LHenvPC_MLMResTab[,1],RZenvPC_MLMResTab[,1],
     xlab = "effect on annual-perennial",ylab = "effect on rhizome",
     main = "env. variables on the evolution of annuality or rhizome",
     col = LHRZCol,pch = 19,asp = 1,ylim = c(-1,1),xaxt = 'n', yaxt = 'n')
abline(h = 0,v = 0, col = "black",lty =2)
legend("topright",pch = 19, col = c("forestgreen","orange","darkred","pink","purple","blue"),
       legend = c("sig. for annual-perennial (+)",
                  "sig. for annual-perennial (-)",
                  "sig. for rhizome (+)",
                  "sig. for rhizome (-)",
                  "sig. for both (+-)",
                  "sig. for both (-+)"))
text(c(-1,1,-1),c(-1,-1,1),c("nonrhizomatous\nperennial", "annual","rhizomatous"),pos = c(4,2,4))
mtext(text = round(exp(seq(-1,1,0.25)),2), side = 1,at = seq(-1,1,0.25),line = .5)
mtext(text = round(exp(seq(-1,1,0.25)),2), side = 2,at = seq(-1,1,0.25),las = 2,line = .5)

asreml.options(maxit = 30,verbose = F)
testMLM_LHRZ = matrix(NA,10,12)
testMLM_LHRZ[1,] = fit_model_binomial(dat1_LH,phyloKMat,"lifeHistoryNumeric",c("GRAVY","MW","Density","NC","Cost","GC"))
testMLM_LHRZ[2,] = fit_model_binomial(dat2_LH,phyloKMat,"lifeHistoryNumeric",c("GRAVY","MW","Density","NC","Cost","GC"))
testMLM_LHRZ[3,] = fit_model_binomial(dat3_LH,phyloKMat,"lifeHistoryNumeric",c("GRAVY","MW","Density","NC","Cost","GC"))
testMLM_LHRZ[4,] = fit_model_binomial(dat4_LH,phyloKMat,"lifeHistoryNumeric",c("GRAVY","MW","Density","NC","Cost","GC"))
testMLM_LHRZ[5,] = fit_model_binomial(dat5_LH,phyloKMat,"lifeHistoryNumeric",c("GRAVY","MW","Density","NC","Cost","GC"))
testMLM_LHRZ[6,] = fit_model_binomial(dat1_RZ,phyloKMat,"rhizome_PAV",c("GRAVY","MW","Density","NC","Cost","GC"))
testMLM_LHRZ[7,] = fit_model_binomial(dat2_RZ,phyloKMat,"rhizome_PAV",c("GRAVY","MW","Density","NC","Cost","GC"))
testMLM_LHRZ[8,] = fit_model_binomial(dat3_RZ,phyloKMat,"rhizome_PAV",c("GRAVY","MW","Density","NC","Cost","GC"))
testMLM_LHRZ[9,] = fit_model_binomial(dat4_RZ,phyloKMat,"rhizome_PAV",c("GRAVY","MW","Density","NC","Cost","GC"))
testMLM_LHRZ[10,] = fit_model_binomial(dat5_RZ,phyloKMat,"rhizome_PAV",c("GRAVY","MW","Density","NC","Cost","GC"))

rownames(testMLM_LHRZ) = paste(rep(c("all OGs","abundant OGs","leaf abundant OGs",
                                     "root abundant OGs","low-abundance OGs"),2),
                               rep(c("life History","rhizome"),each = 5))
colnames(testMLM_LHRZ) = paste(rep(c("coeff","p"),each = 6),
                               rep(c("GRAVY","MW","Density","NC","Cost","GC"),2),sep = "_")

fwrite(testMLM_LHRZ,"/workdir/sh2246/p_phyloGWAS/output/genomicFeatureAssociation_LHRZ_MLM.txt",
       sep = "\t",quote = F,row.names = T)

testMLM_LHRZ = as.matrix(read.delim("/workdir/sh2246/p_phyloGWAS/output/genomicFeatureAssociation_LHRZ_MLM.txt",header = T,row.names = 1))


In [225]:
lifeHistory_perm = dat1_LH$assemblyID
set.seed(123)
for (i in 1:1000){
    # trait vector
    trait_test = dat1_LH$lifeHistoryNumeric
    names(trait_test) = dat1_LH$assemblyID
    subTre = keep.tip(spTre.rooted.resolved,as.character(dat1_LH$assemblyID))

    # rate estimates and simulation
    rate_test = ratematrix(subTre, trait_test)
    trait_sim = sim.char(subTre,rate_test,nsim = 1)
    lifeHistory_perm = data.frame(lifeHistory_perm,sort(trait_test)[rank(trait_sim[,,1])])    
}
colnames(lifeHistory_perm) = c("assemblyID",paste0("lifeHistory","_",1:1000))

rhizome_perm = dat1_RZ$assemblyID
set.seed(123)
for (i in 1:1000){
    # trait vector
    trait_test = dat1_RZ$rhizome_PAV
    names(trait_test) = dat1_RZ$assemblyID
    subTre = keep.tip(spTre.rooted.resolved,as.character(dat1_RZ$assemblyID))

    # rate estimates and simulation
    rate_test = ratematrix(subTre, trait_test)
    trait_sim = sim.char(subTre,rate_test,nsim = 1)
    rhizome_perm = data.frame(rhizome_perm,sort(trait_test)[rank(trait_sim[,,1])])    
}
colnames(rhizome_perm) = c("assemblyID",paste0("rhizomeSim","_",1:1000))

testMLM_LHRZ_perm = list()
asreml.options(maxit = 50,verbose = F)
tmpDat = merge(dat1_LH,lifeHistory_perm,by = 'assemblyID')
tmpName = colnames(tmpDat)[grep("lifeHistory_",colnames(tmpDat))]
testMLM_LHRZ_perm[[1]] = mclapply(tmpName,
                            function(x) fit_model_binomial(tmpDat,phyloKMat,x,c("GRAVY","MW","Density","NC","Cost","GC")),
                                 mc.cores = 20)
testMLM_LHRZ_perm[[1]] = t(simplify2array(testMLM_LHRZ_perm[[1]]))
testMLM_LHRZ_perm[[1]] = as.data.frame(testMLM_LHRZ_perm[[1]])
rownames(testMLM_LHRZ_perm[[1]]) = tmpName

asreml.options(maxit = 50,verbose = F)
tmpDat = merge(dat3_LH,lifeHistory_perm,by = 'assemblyID')
tmpName = colnames(tmpDat)[grep("lifeHistory_",colnames(tmpDat))]
testMLM_LHRZ_perm[[2]] = mclapply(tmpName,
                            function(x) fit_model_binomial(tmpDat,phyloKMat,x,c("GRAVY","MW","Density","NC","Cost","GC")),
                                 mc.cores = 20)
testMLM_LHRZ_perm[[2]] = t(simplify2array(testMLM_LHRZ_perm[[2]]))
testMLM_LHRZ_perm[[2]] = as.data.frame(testMLM_LHRZ_perm[[2]])
rownames(testMLM_LHRZ_perm[[2]]) = tmpName

asreml.options(maxit = 50,verbose = F)
tmpDat = merge(dat4_LH,lifeHistory_perm,by = 'assemblyID')
tmpName = colnames(tmpDat)[grep("lifeHistory_",colnames(tmpDat))]
testMLM_LHRZ_perm[[3]] = mclapply(tmpName,
                            function(x) fit_model_binomial(tmpDat,phyloKMat,x,c("GRAVY","MW","Density","NC","Cost","GC")),
                                 mc.cores = 20)
testMLM_LHRZ_perm[[3]] = t(simplify2array(testMLM_LHRZ_perm[[3]]))
testMLM_LHRZ_perm[[3]] = as.data.frame(testMLM_LHRZ_perm[[3]])
rownames(testMLM_LHRZ_perm[[3]]) = tmpName

asreml.options(maxit = 50,verbose = F)
tmpDat = merge(dat5_LH,lifeHistory_perm,by = 'assemblyID')
tmpName = colnames(tmpDat)[grep("lifeHistory_",colnames(tmpDat))]
testMLM_LHRZ_perm[[4]] = mclapply(tmpName,
                            function(x) fit_model_binomial(tmpDat,phyloKMat,x,c("GRAVY","MW","Density","NC","Cost","GC")),
                                 mc.cores = 20)
testMLM_LHRZ_perm[[4]] = t(simplify2array(testMLM_LHRZ_perm[[4]]))
testMLM_LHRZ_perm[[4]] = as.data.frame(testMLM_LHRZ_perm[[4]])
rownames(testMLM_LHRZ_perm[[4]]) = tmpName


asreml.options(maxit = 50,verbose = F)
tmpDat = merge(dat1_RZ,rhizome_perm,by = 'assemblyID')
tmpName = colnames(tmpDat)[grep("rhizomeSim",colnames(tmpDat))]
testMLM_LHRZ_perm[[5]] = mclapply(tmpName,
                            function(x) fit_model_binomial(tmpDat,phyloKMat,x,c("GRAVY","MW","Density","NC","Cost","GC")),
                                 mc.cores = 20)
testMLM_LHRZ_perm[[5]] = t(simplify2array(testMLM_LHRZ_perm[[5]]))
testMLM_LHRZ_perm[[5]] = as.data.frame(testMLM_LHRZ_perm[[5]])
rownames(testMLM_LHRZ_perm[[5]]) = tmpName

asreml.options(maxit = 50,verbose = F)
tmpDat = merge(dat3_RZ,rhizome_perm,by = 'assemblyID')
tmpName = colnames(tmpDat)[grep("rhizomeSim",colnames(tmpDat))]
testMLM_LHRZ_perm[[6]] = mclapply(tmpName,
                            function(x) fit_model_binomial(tmpDat,phyloKMat,x,c("GRAVY","MW","Density","NC","Cost","GC")),
                                 mc.cores = 20)
testMLM_LHRZ_perm[[6]] = t(simplify2array(testMLM_LHRZ_perm[[6]]))
testMLM_LHRZ_perm[[6]] = as.data.frame(testMLM_LHRZ_perm[[6]])
rownames(testMLM_LHRZ_perm[[6]]) = tmpName

asreml.options(maxit = 50,verbose = F)
tmpDat = merge(dat4_RZ,rhizome_perm,by = 'assemblyID')
tmpName = colnames(tmpDat)[grep("rhizomeSim",colnames(tmpDat))]
testMLM_LHRZ_perm[[7]] = mclapply(tmpName,
                            function(x) fit_model_binomial(tmpDat,phyloKMat,x,c("GRAVY","MW","Density","NC","Cost","GC")),
                                 mc.cores = 20)
testMLM_LHRZ_perm[[7]] = t(simplify2array(testMLM_LHRZ_perm[[7]]))
testMLM_LHRZ_perm[[7]] = as.data.frame(testMLM_LHRZ_perm[[7]])
rownames(testMLM_LHRZ_perm[[7]]) = tmpName

asreml.options(maxit = 50,verbose = F)
tmpDat = merge(dat5_RZ,rhizome_perm,by = 'assemblyID')
tmpName = colnames(tmpDat)[grep("rhizomeSim",colnames(tmpDat))]
testMLM_LHRZ_perm[[8]] = mclapply(tmpName,
                            function(x) fit_model_binomial(tmpDat,phyloKMat,x,c("GRAVY","MW","Density","NC","Cost","GC")),
                                 mc.cores = 20)
testMLM_LHRZ_perm[[8]] = t(simplify2array(testMLM_LHRZ_perm[[8]]))
testMLM_LHRZ_perm[[8]] = as.data.frame(testMLM_LHRZ_perm[[8]])
rownames(testMLM_LHRZ_perm[[8]]) = tmpName


saveRDS(testMLM_LHRZ_perm,"/workdir/sh2246/p_phyloGWAS/output/permMLM_LHRZ.rds")

testMLM_LHRZ_perm = readRDS("/workdir/sh2246/p_phyloGWAS/output/permMLM_LHRZ.rds")

emp_p_partial_LHRZ = c()
perm_coef_avg_LHRZ = c()
perm_coef_se_LHRZ = c()
for(i in 1:8){
    tmp1 = rowSums(t(testMLM_LHRZ_perm[[i]][,1:6])<rep(testMLM_LHRZ[i,1:6],nrow(testMLM_LHRZ_perm[[i]])))/nrow(testMLM_LHRZ_perm[[i]])
    tmp2 = rowSums(t(testMLM_LHRZ_perm[[i]][,1:6])>rep(testMLM_LHRZ[i,1:6],nrow(testMLM_LHRZ_perm[[i]])))/nrow(testMLM_LHRZ_perm[[i]])
    emp_p_partial_LHRZ = rbind(emp_p_partial_LHRZ,apply(cbind(tmp1,tmp2),1,function(x) min(c(1,2*min(x)))))
    perm_coef_avg_LHRZ = rbind(perm_coef_avg_LHRZ,apply(testMLM_LHRZ_perm[[i]][,1:6],2,mean))
    perm_coef_se_LHRZ = rbind(perm_coef_se_LHRZ,apply(testMLM_LHRZ_perm[[i]][,1:6],2,function(x) sd(x)))
}


emp_p_partial_LHRZ
perm_coef_avg_LHRZ
perm_coef_se_LHRZ

options(repr.plot.width=10, repr.plot.height=10)
par(mfrow = c(2,1))
par(mar = c(0,5,0,3))
sig = unlist(testMLM_LHRZ[c(1,3:5),7:12])
sig2 = as.vector(emp_p_partial_LHRZ[1:4,])
print(table(sig<.05,sig2<.05))
seVec = as.vector(perm_coef_se_LHRZ[1:4,])
sigLab = ifelse(sig<.05,'*','')
sigLab[sig<.01] = "**"
sigLab[sig<.005] = "***"
labs = paste0(round(unlist(exp(-testMLM_LHRZ[c(1,3:5),1:6])),2),sigLab)
labs[sig> 0.05] = ""
bp1 = barplot(-testMLM_LHRZ[c(1,3:5),1:6],beside = T,density = ifelse(sig<.05,100,0),horiz = F,
                 yaxt = 'n',angle = 45,ylim = c(-.9,.9),names.arg = rep(NA,6),
              col = alpha(c("royalblue","forestgreen","darkgoldenrod","grey"),.5),
                 border = c("royalblue","forestgreen","darkgoldenrod","grey"))
legend("topright",bty = 'n', fill = alpha(c("royalblue","forestgreen","darkgoldenrod","grey"),.5),
       border = c("royalblue","forestgreen","darkgoldenrod","grey"),
       legend = c("all OGs","leaf abundant OGs","root abundant OGs","low-abundance OGs"),cex = 1.1)
text(y = -testMLM_LHRZ[c(1,3:5),1:6] + ifelse(-testMLM_LHRZ[c(1,3:5),1:6]>0,1,-1)*seVec*1.5,
     x = unlist(bp1),labels = labs,cex = 1.1)
abline(h= 0)
mtext("Life History",at = 0, line = 3, cex =2,side = 2)
mtext(c("Perennial","Annual"),at = c(-0.5,0.5), line = -2, cex =1.5,side = 2,las = 1)
arrows(x0 = unlist(bp1),x1 = unlist(bp1),
           y0 = -testMLM_LHRZ[c(1,3:5),1:6]-seVec,
           y1 = -testMLM_LHRZ[c(1,3:5),1:6]+seVec,
           code = 3, angle = 90, length = .025, col = c("royalblue","forestgreen","darkgoldenrod","grey"))

par(mar = c(4,5,.5,3))
sig = unlist(testMLM_LHRZ[c(6,8:10),7:12])
sig2 = as.vector(emp_p_partial_LHRZ[5:8,])
print(table(sig<.05,sig2<.05))
seVec = as.vector(perm_coef_se_LHRZ[5:8,])
sigLab = ifelse(sig<.05,'*','')
sigLab[sig<.01] = "**"
sigLab[sig<.005] = "***"
labs = paste0(round(unlist(exp(testMLM_LHRZ[c(6,8:10),1:6])),2),sigLab)
labs[sig> 0.05] = ""
bp2 = barplot(testMLM_LHRZ[c(6,8:10),1:6],beside = T,density = ifelse(sig<.05,100,0),horiz = F,
                 yaxt = 'n',angle = 45,ylim = c(-.9,.9),names.arg = rep(NA,6),
              col = alpha(c("royalblue","forestgreen","darkgoldenrod","grey"),.5),
                 border = c("royalblue","forestgreen","darkgoldenrod","grey"))
text(y = testMLM_LHRZ[c(6,8:10),1:6] + ifelse(testMLM_LHRZ[c(6,8:10),1:6]>0,1,-1)*1.5,
     x = unlist(bp2),labels = labs,cex = 1.1)
arrows(x0 = unlist(bp2),x1 = unlist(bp2),
           y0 = testMLM_LHRZ[c(6,8:10),1:6]-seVec,
           y1 = testMLM_LHRZ[c(6,8:10),1:6]+seVec,
           code = 3, angle = 90, length = .025, col = c("royalblue","forestgreen","darkgoldenrod","grey"))
abline(h= 0)
mtext(c("Hydropathy","Molecular Weight","Density","Nitrogen/Carbon","Energetic Cost","GC%"),
      at = apply(bp2,2,mean),1,line = 2,cex = 1.25)
mtext("Rhizome",at = 0, line = 3, cex =2,side = 2)
mtext(c("Absence","Presence"),at = c(-0.5,0.5), line = -2, cex =1.5,side = 2,las = 1)

png("/workdir/sh2246/p_phyloGWAS/output/figure/lifeHistory/genomeFeatureModel_OR.png",
    width = 8.7*2.5,height = 8.7*1.5,units = "cm",res = 600,pointsize = 8)
par(mfrow = c(2,1))
par(mar = c(0,5,0,3))
sig = unlist(testMLM_LHRZ[c(1,3:5),7:12])
# sig2 = as.vector(emp_p_partial_LHRZ[1:4,])
# print(table(sig<.05,sig2<.05))
seVec = as.vector(perm_coef_se_LHRZ[1:4,])
sigLab = ifelse(sig<.05,'*','')
sigLab[sig<.01] = "**"
sigLab[sig<.005] = "***"
labs = paste0(round(unlist(exp(-testMLM_LHRZ[c(1,3:5),1:6])),2),sigLab)
labs[sig> 0.05] = ""
bp1 = barplot(-testMLM_LHRZ[c(1,3:5),1:6],beside = T,density = ifelse(sig<.05,100,0),horiz = F,
                 yaxt = 'n',angle = 45,ylim = c(-.9,.9),names.arg = rep(NA,6),
              col = alpha(c("royalblue","forestgreen","darkgoldenrod","grey"),.5),
                 border = c("royalblue","forestgreen","darkgoldenrod","grey"))
legend("topright",bty = 'n', fill = alpha(c("royalblue","forestgreen","darkgoldenrod","grey"),.5),
       border = c("royalblue","forestgreen","darkgoldenrod","grey"),
       legend = c("all OGs","leaf abundant OGs","root abundant OGs","low-abundance OGs"),cex = 1.1)
text(y = -testMLM_LHRZ[c(1,3:5),1:6] + ifelse(-testMLM_LHRZ[c(1,3:5),1:6]>0,1,-1)*seVec*1.5,
     x = unlist(bp1),labels = labs,cex = 1.1)
abline(h= 0)
mtext("Life History",at = 0, line = 3, cex =2,side = 2)
mtext(c("Perennial","Annual"),at = c(-0.7,0.7), line = -2, cex =1.5,side = 2,las = 1)
arrows(x0 = unlist(bp1),x1 = unlist(bp1),
           y0 = -testMLM_LHRZ[c(1,3:5),1:6]-seVec,
           y1 = -testMLM_LHRZ[c(1,3:5),1:6]+seVec,
           code = 3, angle = 90, length = .025, col = c("royalblue","forestgreen","darkgoldenrod","grey"))

par(mar = c(4,5,.5,3))
sig = unlist(testMLM_LHRZ[c(6,8:10),7:12])
# sig2 = as.vector(emp_p_partial_LHRZ[5:8,])
# print(table(sig<.05,sig2<.05))
seVec = as.vector(perm_coef_se_LHRZ[5:8,])
sigLab = ifelse(sig<.05,'*','')
sigLab[sig<.01] = "**"
sigLab[sig<.005] = "***"
labs = paste0(round(unlist(exp(testMLM_LHRZ[c(6,8:10),1:6])),2),sigLab)
labs[sig> 0.05] = ""
bp2 = barplot(testMLM_LHRZ[c(6,8:10),1:6],beside = T,density = ifelse(sig<.05,100,0),horiz = F,
                 yaxt = 'n',angle = 45,ylim = c(-.9,.9),names.arg = rep(NA,6),
              col = alpha(c("royalblue","forestgreen","darkgoldenrod","grey"),.5),
                 border = c("royalblue","forestgreen","darkgoldenrod","grey"))
text(y = testMLM_LHRZ[c(6,8:10),1:6] + ifelse(testMLM_LHRZ[c(6,8:10),1:6]>0,1,-1)*seVec*c(1.6,2.05),
     x = unlist(bp2),labels = labs,cex = 1.1)
arrows(x0 = unlist(bp2),x1 = unlist(bp2),
           y0 = testMLM_LHRZ[c(6,8:10),1:6]-seVec,
           y1 = testMLM_LHRZ[c(6,8:10),1:6]+seVec,
           code = 3, angle = 90, length = .025, col = c("royalblue","forestgreen","darkgoldenrod","grey"))
abline(h= 0)
mtext(c("Hydropathy","Molecular Weight","Density","Nitrogen/Carbon","Energetic Cost","GC%"),
      at = apply(bp2,2,mean),1,line = 2,cex = 1.25)
mtext("Rhizome",at = 0, line = 3, cex =2,side = 2)
mtext(c("Absence","Presence"),at = c(-0.7,0.7), line = -2, cex =1.5,side = 2,las = 1)
dev.off()